# Inside a model: what did we actually download?

This notebook looks inside the package behind a small language model. We will answer questions like these:

- How many **parameters** does the model have?
- How many weight arrays and matrices are stored?
- What are the **weights** themselves?
- What does it mean that the model has **open weights**?
- What files does WebLLM download into the browser?

A **parameter** is a number learned during training. A **weight** is a parameter that tells the model how strongly one value should affect another.

## Step 1: Get set up

This notebook does not run a model, so it does not need WebGPU or WebLLM. Instead, it looks up public information from **Hugging Face** (the website where models are shared) about the original model and the converted WebLLM package.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `pandas` (as `pd`) makes and filters tables.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.

In [ ]:
import json
import pandas as pd
import pyjs

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Helpers for fetching and printing

- `fetch_json()` downloads a page of data from a web address. It uses the browser's own `fetch`, then turns the JSON text into Python dictionaries with `json.loads`.
- The `format_` helpers print big numbers in a readable way, like `999,885,952` or `1.00 billion`.

In [ ]:
async def fetch_json(url):
    """Download JSON from a web address and return it as Python data."""
    response = await js.fetch(url)
    if response.status != 200:
        raise RuntimeError(f"Could not load {url} (status {response.status}). Check the repo names in Step 2.")
    return json.loads(await response.text())


def format_int(value):
    return f"{int(value):,}"


def format_millions(value):
    return f"{value / 1_000_000:.1f} million"


def format_billions(value):
    return f"{value / 1_000_000_000:.2f} billion"


print("Helpers are ready.")

## Step 2: Pick a model family

We need two Hugging Face **repositories** (a repository, or repo, is a model's folder of files) for each model:

1. the **original model** from the model maker
2. the **converted model** from `mlc-ai`, which is the packaged WebLLM version

Put one of the names from `MODEL_OPTIONS` into `MODEL_FAMILY`. The default is Gemma 3 1B, one of the main teaching models in this project.

In [ ]:
MODEL_FAMILY = "Gemma 3 1B Instruct"

MODEL_OPTIONS = {
    "Gemma 3 1B Instruct": {
        "original_repo": "google/gemma-3-1b-it",
        "converted_repo": "mlc-ai/gemma3-1b-it-q4f16_1-MLC",
    },
    "Llama 3.2 1B Instruct": {
        "original_repo": "meta-llama/Llama-3.2-1B-Instruct",
        "converted_repo": "mlc-ai/Llama-3.2-1B-Instruct-q4f16_1-MLC",
    },
    "Qwen 2.5 0.5B Instruct": {
        "original_repo": "Qwen/Qwen2.5-0.5B-Instruct",
        "converted_repo": "mlc-ai/Qwen2.5-0.5B-Instruct-q4f16_1-MLC",
    },
}

if MODEL_FAMILY not in MODEL_OPTIONS:
    print(f"{MODEL_FAMILY!r} is not in MODEL_OPTIONS. Pick one of: {', '.join(MODEL_OPTIONS)}")
else:
    model_info = MODEL_OPTIONS[MODEL_FAMILY]
    print("Model family:   " + MODEL_FAMILY)
    print("Original repo:  " + model_info["original_repo"])
    print("WebLLM package: " + model_info["converted_repo"])

## Step 3: Load public metadata

**Metadata** is information about something, rather than the thing itself. The next cell fetches four pieces:

- the original model's information page
- the converted WebLLM package's information page
- the WebLLM chat config, which describes the model's shape
- the WebLLM **tensor inventory**, which lists every stored weight array, its shape, and which file it lives in

In [ ]:
original_api = "https://huggingface.co/api/models/" + model_info["original_repo"]
converted_api = "https://huggingface.co/api/models/" + model_info["converted_repo"]
chat_config_url = "https://huggingface.co/" + model_info["converted_repo"] + "/resolve/main/mlc-chat-config.json"
tensor_cache_url = "https://huggingface.co/" + model_info["converted_repo"] + "/resolve/main/tensor-cache.json"

original_info = await fetch_json(original_api)
converted_info = await fetch_json(converted_api)
chat_config = await fetch_json(chat_config_url)
tensor_cache = await fetch_json(tensor_cache_url)

print("Metadata loaded.")

## Step 4: Start with the biggest numbers

A first summary of the model and its WebLLM package.

- **BF16** is the usual way the original weights are stored: 16 bits (2 bytes) per parameter.
- WebLLM's package is **quantized**: stored with fewer bits per parameter, which makes it much smaller.

In [ ]:
original_parameters = original_info.get("safetensors", {}).get("total")
quantized_bytes = tensor_cache["metadata"]["ParamBytes"]
bits_per_param = tensor_cache["metadata"]["BitsPerParam"]
bf16_bytes = original_parameters * 2 if original_parameters else None
compression_ratio = (bf16_bytes / quantized_bytes) if bf16_bytes else None

summary_rows = [
    {"quantity": "Original parameter count", "value": format_int(original_parameters) if original_parameters else "unknown"},
    {"quantity": "That is about", "value": format_billions(original_parameters) if original_parameters else "unknown"},
    {"quantity": "Quantized WebLLM weight size", "value": format_millions(quantized_bytes) + " bytes"},
    {"quantity": "Average bits per parameter", "value": f"{bits_per_param:.2f} bits"},
    {"quantity": "Approximate original BF16 size", "value": format_millions(bf16_bytes) + " bytes" if bf16_bytes else "unknown"},
    {"quantity": "Approximate compression", "value": f"{compression_ratio:.2f}x smaller than BF16" if compression_ratio else "unknown"},
]

pd.DataFrame(summary_rows)

## Step 5: What does a parameter mean?

A parameter is one learned number. But one number by itself is not very interesting. The real action comes from **large tables of numbers** (matrices) that transform one list of numbers (a vector) into another.

The next cell is a tiny toy example, not a real model weight file. Each output number is made by multiplying each input number by a weight and adding them up.

In [ ]:
toy_input = [1.0, 0.5, -0.5]
toy_weights = [
    [0.2, -0.4, 0.8],
    [1.1, 0.3, -0.2],
]


def matvec(matrix, vector):
    output = []
    for row in matrix:
        total = 0.0
        for weight, value in zip(row, vector):
            total += weight * value
        output.append(total)
    return output


toy_output = matvec(toy_weights, toy_input)
print("Input vector: ", toy_input)
print("Weight matrix:", toy_weights)
print("Output vector:", toy_output)

In plain language, each weight says how strongly one input number should push an output number up or down. A real language model does this with much larger vectors and many more layers.

## Step 6: Read the model's architecture

A model's **architecture** is its overall shape. The WebLLM chat config tells us how many **layers** it has (steps the numbers pass through, one after another), how wide its hidden state is (how many numbers represent each token inside the model), how many attention heads it uses, and how big its vocabulary (list of tokens it knows) is.

In [ ]:
text_config = chat_config["model_config"].get("text_config", chat_config["model_config"])

architecture_rows = [
    {"field": "model_type", "value": chat_config.get("model_type")},
    {"field": "quantization", "value": chat_config.get("quantization")},
    {"field": "num_hidden_layers", "value": text_config.get("num_hidden_layers")},
    {"field": "hidden_size", "value": text_config.get("hidden_size")},
    {"field": "intermediate_size", "value": text_config.get("intermediate_size")},
    {"field": "num_attention_heads", "value": text_config.get("num_attention_heads")},
    {"field": "num_key_value_heads", "value": text_config.get("num_key_value_heads")},
    {"field": "head_dim", "value": text_config.get("head_dim")},
    {"field": "vocab_size", "value": chat_config.get("vocab_size")},
    {"field": "context_window_size", "value": chat_config.get("context_window_size")},
]

pd.DataFrame(architecture_rows)

## Step 7: Flatten the tensor inventory

The converted WebLLM package stores weights in **shard** files such as `params_shard_0.bin`, `params_shard_1.bin`, and so on (a big file split into pieces). Inside those files are many named tensors.

A **tensor** is a container of numbers. A vector is a 1-dimensional tensor. A matrix is a 2-dimensional tensor. The **rank** of a tensor is how many dimensions it has.

The inventory is nested (shards, then tensors inside each shard), so this cell flattens it into one row per tensor.

In [ ]:
tensor_rows = []
for shard in tensor_cache.get("records", []):
    for record in shard.get("records", []):
        shape = record.get("shape", [])
        tensor_rows.append({
            "shard": shard.get("dataPath"),
            "name": record.get("name"),
            "shape": shape,
            "rank": len(shape),
            "dtype": record.get("dtype"),
            "nbytes": record.get("nbytes"),
        })

tensors = pd.DataFrame(tensor_rows)
print("Tensors found:", len(tensors))
print("Shard files found:", len(tensor_cache.get("records", [])))

## Step 8: How many matrices are in the package?

Most of the big learned transformations live in 2-dimensional tensors, which are matrices. Some other tensors are 1-dimensional vectors, such as normalization weights.

In [ ]:
pd.DataFrame([
    {"kind": "2D tensors (matrices)", "count": (tensors["rank"] == 2).sum()},
    {"kind": "1D tensors (vectors)", "count": (tensors["rank"] == 1).sum()},
    {"kind": "Other tensor ranks", "count": (~tensors["rank"].isin([1, 2])).sum()},
    {"kind": "Total tensors", "count": len(tensors)},
])

## Step 9: What kinds of tensors are these?

Tensor names tell us roughly what job they do:

- `embed_tokens`: the token embedding table, which turns each token into a vector
- `self_attn`: **self-attention**, where each token looks at the other tokens around it
- `mlp`: the **feed-forward** block, where each token's numbers are transformed on their own
- `norm`: a **normalization** step that keeps the numbers in a steady range
- `lm_head`: the output head, which turns the final numbers into scores for each possible next token

The next cell sorts every tensor into one of these groups and counts them.

In [ ]:
def tensor_category(name):
    if "embed_tokens" in name:
        return "token embeddings"
    if "self_attn" in name:
        return "attention"
    if ".mlp." in name:
        return "feed-forward (MLP)"
    if "norm" in name:
        return "normalization"
    if "lm_head" in name:
        return "output head"
    return "other"


tensors["category"] = tensors["name"].apply(tensor_category)
tensors["category"].value_counts().rename_axis("category").reset_index(name="tensor_count")

## Step 10: Look at a few real tensor names and shapes

These names may look intimidating at first, but they are useful clues. The numbers like `layers.0` tell you which layer the tensor belongs to, and `shape` tells you how many rows and columns of numbers it holds.

In [ ]:
tensors[["name", "shape", "dtype", "nbytes", "shard"]].head(20)

## Step 11: What files are in the open-weight package?

When we say we downloaded the model, we did **not** download a magical black box. We downloaded a package of ordinary files. The converted WebLLM repository tells us what they are.

In [ ]:
def file_kind(name):
    if name.startswith("params_shard_") and name.endswith(".bin"):
        return "weight shard"
    if name == "tensor-cache.json":
        return "tensor inventory"
    if name == "mlc-chat-config.json":
        return "runtime chat config"
    if "tokenizer" in name or "added_tokens" in name:
        return "tokenizer file"
    return "other"


file_rows = []
for item in converted_info.get("siblings", []):
    name = item.get("rfilename")
    file_rows.append({"file": name, "kind": file_kind(name)})

pd.DataFrame(file_rows)

## Step 12: What does "open weights" mean here?

A plain-language definition:

- The **weights** are the learned numbers of the model.
- **Open weights** means those learned numbers have been published so other people can download and run them.
- That does **not** always mean every part of the training process is open. The training code, data, and license may be more or less open depending on the model family.

In this notebook, the original model repository holds the original published weights, while the `mlc-ai` repository holds a converted and compressed package that WebLLM can run in the browser.

## Step 13: Why does the WebLLM package look different from the original model?

The WebLLM package has a different job. It is meant to run inside a browser, so it is usually:

- **quantized**, which means weights are stored with fewer bits
- **sharded**, which means split across several files
- paired with extra config files that tell WebLLM how to load and use the package

That is why you see files like `params_shard_3.bin` and `tensor-cache.json` instead of one giant `model.safetensors` file. The next cell compares the two repositories.

In [ ]:
original_files = [item.get("rfilename") for item in original_info.get("siblings", [])]
converted_files = [item.get("rfilename") for item in converted_info.get("siblings", [])]

pd.DataFrame([
    {"package": "Original model repo", "file_count": len(original_files), "examples": ", ".join(original_files[:6])},
    {"package": "Converted WebLLM repo", "file_count": len(converted_files), "examples": ", ".join(converted_files[:6])},
])

## Step 14: A short summary

If you had to explain this model package to a friend, you could say:

1. A language model is mostly a huge pile of learned numbers.
2. Those numbers are stored in tensors, and many of the most important tensors are matrices.
3. The weights are "open" because people can download them.
4. WebLLM downloads a browser-ready version of those weights, compressed and split into shard files.
5. The tokenizer files teach the model how to break text into tokens.
6. The config files teach WebLLM how to load the package and chat with it.

So when we say "we downloaded a model," we really mean: we downloaded a carefully packaged collection of numbers plus the files needed to interpret them.

## Your turn

Change `MODEL_FAMILY` in Step 2, then run Steps 3 to 13 again to compare two model families. Which one has more layers? Which one has a larger vocabulary? Which one needs more shard files?